# 03 — Indexación y recuperación semántica con ChromaDB

**Proyecto:** Chem RAG Assistant  
**Repositorio:** https://github.com/Jesusrodriguezf90/chem-rag-assistant  
**Fase:** Prototipado del pipeline de recuperación

---

Este notebook cubre la tercera fase del pipeline RAG: la indexación de los
embeddings generados en `02_embeddings.ipynb` en una base de datos vectorial
ChromaDB y la validación del sistema de recuperación semántica.

**Entrada:** `data/embeddings/PMC10967698_embeddings.npy` + `PMC10967698_chunks.json`  
**Salida:** base de datos ChromaDB persistida en `data/chroma_db/` lista para el pipeline de generación

In [ ]:
"""
Notebook: 03_recuperacion.ipynb

Objetivo:
    Indexar los embeddings generados con BGE-M3 en ChromaDB y validar
    el sistema de recuperación semántica mediante un conjunto de preguntas
    de evaluación con progresión de complejidad (básica, intermedia, avanzada),
    siguiendo el estándar de la industria para evaluación de pipelines RAG.

    Este notebook cubre:
      1. Carga de embeddings y chunks persistidos
      2. Instalación de dependencias
      3. Definición de rutas y parámetros
      4. Carga de embeddings y chunks
      5. Inicialización y persistencia de ChromaDB
      6. Indexación de embeddings con metadatos
      7. Validación del índice
      8. Carga del modelo de embeddings para consultas
      9. Definición de preguntas de evaluación con ground truth
      10. Recuperación semántica con top-k=4
      11. Evaluación con preguntas de complejidad progresiva
      12. Resumen cuantitativo de resultados

Fuente de datos:
    Büchele WRE, Schlachta TP, Gebendorfer AL, Pamperin J, Richter LF,
    Sauer MJ, Prokop A, Kühn FE. Synthesis, characterization, and biomedical
    evaluation of ethylene-bridged tetra-NHC Pd(ii), Pt(ii) and Au(iii)
    complexes, with apoptosis-inducing properties in cisplatin-resistant
    neuroblastoma cells. Frontiers in Chemistry. 2024.
    PMC: https://pmc.ncbi.nlm.nih.gov/articles/PMC10967698/

    Documento utilizado exclusivamente con fines de investigación y desarrollo.
    No se distribuye ni se incluye en el repositorio.

Autor:   Jesús Rodríguez
Fecha:   2026-05-05
Versión: 1.0.1
"""

'\nNotebook: 03_recuperacion.ipynb\n\nObjetivo:\n    Indexar los embeddings generados con BGE-M3 en ChromaDB y validar\n    el sistema de recuperación semántica mediante un conjunto de preguntas\n    de evaluación con progresión de complejidad (básica, intermedia, avanzada),\n    siguiendo el estándar de la industria para evaluación de pipelines RAG.\n\n    Este notebook cubre:\n      1. Carga de embeddings y chunks persistidos\n      2. Instalación de dependencias\n      3. Definición de rutas y parámetros\n      4. Carga de embeddings y chunks\n      5. Inicialización y persistencia de ChromaDB\n      6. Indexación de embeddings con metadatos\n      7. Validación del índice\n      8. Carga del modelo de embeddings para consultas\n      9. Definición de preguntas de evaluación con ground truth\n      10. Recuperación semántica con top-k=4\n      11. Evaluación con preguntas de complejidad progresiva\n      12. Resumen cuantitativo de resultados\n\nFuente de datos:\n    Büchele WRE, Sc

## 1. Configuración del entorno

In [ ]:
# Detección del entorno de ejecución (Colab vs local)
try:
    from google.colab import drive
    IN_COLAB = True
    print("Entorno detectado: Google Colab")
except ImportError:
    IN_COLAB = False
    print("Entorno detectado: local")

Entorno detectado: Google Colab


In [ ]:
# Montaje de Google Drive (solo en Colab)
if IN_COLAB:
    drive.mount('/content/drive')
    print("Google Drive montado correctamente")

Mounted at /content/drive
Google Drive montado correctamente


In [ ]:
# Verificación del entorno de ejecución
import sys
import platform

print(f"Python : {sys.version}")
print(f"Sistema: {platform.system()} {platform.release()}")

# Detección de GPU — relevante para la generación de embeddings de consulta
import torch

if torch.cuda.is_available():
    dispositivo = "cuda"
    nombre_gpu = torch.cuda.get_device_name(0)
    print(f"Dispositivo : GPU — {nombre_gpu}")
else:
    dispositivo = "cpu"
    print("Dispositivo : CPU (correcto para prototipado)")

Python : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Sistema: Linux 6.6.113+
Dispositivo : CPU (correcto para prototipado)


## 2. Instalación de dependencias

In [ ]:
# ChromaDB: base de datos vectorial embebida para búsqueda por similitud
# FlagEmbedding: necesario para generar embeddings de las consultas
# con el mismo modelo usado en la indexación (BGE-M3)
%pip install chromadb FlagEmbedding -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

## 3. Definición de rutas y parámetros

In [ ]:
# Librería estándar para manejo de rutas
from pathlib import Path

# Ruta raíz del proyecto en Google Drive
PROYECTO_RAIZ = Path('/content/drive/MyDrive/chem-rag-assistant')

# Rutas de entrada — outputs de 02_embeddings.ipynb
DIR_EMBEDDINGS  = PROYECTO_RAIZ / 'data' / 'embeddings'
RUTA_EMBEDDINGS = DIR_EMBEDDINGS / 'PMC10967698_embeddings.npy'
RUTA_CHUNKS     = DIR_EMBEDDINGS / 'PMC10967698_chunks.json'

# Ruta de persistencia de ChromaDB
DIR_CHROMA = PROYECTO_RAIZ / 'data' / 'chroma_db'
DIR_CHROMA.mkdir(parents=True, exist_ok=True)

# Parámetros de recuperación
# top_k=4 equilibra cobertura semántica y precisión para
# preguntas de complejidad variable sobre documentos científicos
TOP_K             = 4
NOMBRE_COLECCION  = 'chem_rag_pmc10967698'
MODELO_EMBEDDINGS = 'BAAI/bge-m3'

# Validación de existencia de archivos de entrada
assert RUTA_EMBEDDINGS.exists(), (
    f"Embeddings no encontrados: {RUTA_EMBEDDINGS}\n"
    f"Ejecuta primero el notebook 02_embeddings.ipynb"
)
assert RUTA_CHUNKS.exists(), (
    f"Chunks no encontrados: {RUTA_CHUNKS}\n"
    f"Ejecuta primero el notebook 02_embeddings.ipynb"
)

print(f"Embeddings        : {RUTA_EMBEDDINGS.name}")
print(f"Chunks            : {RUTA_CHUNKS.name}")
print(f"ChromaDB          : {DIR_CHROMA}")
print(f"Colección         : {NOMBRE_COLECCION}")
print(f"Top-k             : {TOP_K}")

Embeddings        : PMC10967698_embeddings.npy
Chunks            : PMC10967698_chunks.json
ChromaDB          : /content/drive/MyDrive/chem-rag-assistant/data/chroma_db
Colección         : chem_rag_pmc10967698
Top-k             : 4


## 4. Carga de embeddings y chunks

In [ ]:
# Carga de los embeddings y chunks generados en 02_embeddings.ipynb
import numpy as np
import json

# Embeddings en formato numpy
embeddings = np.load(str(RUTA_EMBEDDINGS))

# Chunks en formato JSON
with open(RUTA_CHUNKS, 'r', encoding='utf-8') as f:
    chunks = json.load(f)

print(f"Embeddings cargados : {embeddings.shape}")
print(f"Chunks cargados     : {len(chunks)}")

# Verificación de coherencia entre embeddings y chunks
assert embeddings.shape[0] == len(chunks), (
    f"Error: número de embeddings ({embeddings.shape[0]}) "
    f"no coincide con número de chunks ({len(chunks)})"
)
print("Coherencia embeddings-chunks: OK")

Embeddings cargados : (77, 1024)
Chunks cargados     : 77
Coherencia embeddings-chunks: OK


## 5. Inicialización de ChromaDB

In [ ]:
# Inicialización de ChromaDB con persistencia en Google Drive
# La persistencia evita reindexar en cada sesión de Colab
import chromadb

cliente = chromadb.PersistentClient(path=str(DIR_CHROMA))

# Eliminar colección existente si hay una previa para evitar duplicados
# en caso de re-ejecución del notebook
try:
    cliente.delete_collection(name=NOMBRE_COLECCION)
    print(f"Colección previa eliminada: {NOMBRE_COLECCION}")
except Exception:
    print(f"No existía colección previa — creando nueva")

# Crear colección con métrica de similitud coseno
# cosine es la métrica correcta para embeddings de BGE-M3
coleccion = cliente.create_collection(
    name=NOMBRE_COLECCION,
    metadata={"hnsw:space": "cosine"},
)

print(f"Colección creada   : {NOMBRE_COLECCION}")
print(f"Métrica            : coseno")

Colección previa eliminada: chem_rag_pmc10967698
Colección creada   : chem_rag_pmc10967698
Métrica            : coseno


## 6. Indexación de embeddings

In [ ]:
# Indexación de todos los chunks y sus embeddings en ChromaDB
# Cada documento se identifica con un ID único basado en su índice
# y se almacena el texto del chunk como metadata para recuperarlo
# directamente sin necesidad de un almacén externo
print("Indexando embeddings en ChromaDB...")

coleccion.add(
    ids=[f"chunk_{i:04d}" for i in range(len(chunks))],
    embeddings=embeddings.tolist(),
    documents=chunks,
    metadatas=[
        {
            "fuente"    : "PMC10967698",
            "indice"    : i,
            "longitud"  : len(chunks[i]),
        }
        for i in range(len(chunks))
    ],
)

print(f"Chunks indexados   : {coleccion.count()}")

Indexando embeddings en ChromaDB...
Chunks indexados   : 77


## 7. Validación del índice

In [ ]:
# Verificación de que el índice contiene el número correcto de documentos
# y que los metadatos se almacenaron correctamente
assert coleccion.count() == len(chunks), (
    f"Error: el índice contiene {coleccion.count()} documentos, "
    f"se esperaban {len(chunks)}"
)

# Inspección de un documento indexado para verificar la estructura
muestra = coleccion.get(
    ids=["chunk_0000"],
    include=["documents", "metadatas"],
)

print("Validación del índice: OK")
print(f"Total documentos    : {coleccion.count()}")
print()
print("Muestra — chunk_0000:")
print("-" * 45)
print(f"Texto    : {muestra['documents'][0][:150]}...")
print(f"Metadata : {muestra['metadatas'][0]}")

Validación del índice: OK
Total documentos    : 77

Muestra — chunk_0000:
---------------------------------------------
Texto    : ## Introduction

N-heterocyclic carbenes (NHCs),  first described in 1991, 1 have found many applications. 2 There are several structural features tha...
Metadata : {'longitud': 902, 'fuente': 'PMC10967698', 'indice': 0}


## 8. Carga del modelo de embeddings para consultas

In [ ]:
# El modelo de embeddings de consulta debe ser idéntico al usado
# en la indexación (BGE-M3) para garantizar que los vectores
# de consulta y documento están en el mismo espacio semántico
from FlagEmbedding import BGEM3FlagModel

print(f"Cargando modelo {MODELO_EMBEDDINGS}...")

modelo = BGEM3FlagModel(
    MODELO_EMBEDDINGS,
    use_fp16=True,
    device=dispositivo,
)

print(f"Modelo cargado en  : {dispositivo.upper()}")

Cargando modelo BAAI/bge-m3...


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Modelo cargado en  : CPU


## 9. Definición de preguntas de evaluación

In [ ]:
# Preguntas de evaluación con progresión de complejidad
# siguiendo el estándar de la industria para evaluación de RAG:
#   - Básicas: recuperación de hechos directos del documento
#   - Intermedias: requieren integración de información de varias secciones
#   - Avanzadas: requieren razonamiento sobre resultados experimentales
#
# Cada pregunta incluye:
#   - ground_truth: respuesta de referencia para evaluar correctitud factual
#   - chunk_esperado: sección del documento donde debería encontrarse la respuesta
#     usado para validar la recuperación — ver resultados en 02b_chunking_eval.ipynb
PREGUNTAS_EVALUACION = [
    # Básicas
    {
        "pregunta"       : "What metals are used in the NHC complexes studied?",
        "nivel"          : "básica",
        "seccion"        : "Introduction / Results",
        "ground_truth"   : "Palladium (Pd), platinum (Pt) and gold (Au).",
        "chunk_esperado" : "Introduction or Synthesis section mentioning Pd, Pt, Au",
    },
    {
        "pregunta"       : "What is the effect of the complexes on cisplatin-resistant neuroblastoma cells?",
        "nivel"          : "básica",
        "seccion"        : "Biological evaluation",
        "ground_truth"   : (
            "The complexes induce apoptosis in cisplatin-resistant "
            "SK-N-AS neuroblastoma cells."
        ),
        "chunk_esperado" : "Biological evaluation section",
    },
    {
        "pregunta"       : "What analytical techniques were used to characterize the compounds?",
        "nivel"          : "básica",
        "seccion"        : "Experimental section",
        "ground_truth"   : (
            "NMR spectroscopy (1H, 13C, 31P), ESI mass spectrometry "
            "and IR spectroscopy."
        ),
        "chunk_esperado" : "General procedures and analytical methods section",
    },
    # Intermedia
    {
        "pregunta"       : "What is the role of the ethylene bridge in the tetra-NHC ligand design?",
        "nivel"          : "intermedia",
        "seccion"        : "Introduction / Synthesis",
        "ground_truth"   : (
            "The ethylene bridge connects two NHC units forming a tetradentate "
            "chelating ligand that stabilizes the metal center."
        ),
        "chunk_esperado" : "Introduction section on NHC ligand design",
    },
    # Avanzada
    {
        "pregunta"       : "How do the cytotoxicity results of the Au(III) complexes compare to cisplatin?",
        "nivel"          : "avanzada",
        "seccion"        : "Biological evaluation / Conclusion",
        "ground_truth"   : (
            "The Au(III) complexes show higher cytotoxicity than cisplatin "
            "in resistant cell lines, overcoming cisplatin resistance."
        ),
        "chunk_esperado" : "Biological evaluation section on cytotoxicity results",
    },
]

print(f"Preguntas de evaluación cargadas: {len(PREGUNTAS_EVALUACION)}")
print("-" * 45)
for item in PREGUNTAS_EVALUACION:
    print(f"  [{item['nivel'].upper()}] {item['pregunta'][:60]}...")

Preguntas de evaluación cargadas: 5
---------------------------------------------
  [BÁSICA] What metals are used in the NHC complexes studied?...
  [BÁSICA] What is the effect of the complexes on cisplatin-resistant n...
  [BÁSICA] What analytical techniques were used to characterize the com...
  [INTERMEDIA] What is the role of the ethylene bridge in the tetra-NHC lig...
  [AVANZADA] How do the cytotoxicity results of the Au(III) complexes com...


## 10. Recuperación semántica

In [ ]:
# Función de recuperación semántica
# Genera el embedding de la consulta con BGE-M3 y recupera
# los top-k chunks más similares de ChromaDB
#
# NOTA PARA MIGRACIÓN A src/retrieval/vector_store.py:
# 1. Añadir try/except con logging para trazabilidad en producción
# 2. Añadir filtrado por metadatos (fuente=) para soporte multi-documento
def recuperar_chunks(pregunta: str, top_k: int = TOP_K) -> dict:
    """Recupera los chunks más relevantes para una pregunta dada."""
    # Generar embedding de la consulta
    resultado = modelo.encode(
        [pregunta],
        batch_size=1,   # Conservador para no agotar memoria en CPU
        max_length=512,  # Límite de tokens antes de truncar la pregunta
        return_dense=True, # Vector denso de 1024d — el que usamos para búsqueda
        return_sparse=False, # Vectores dispersos para búsqueda híbrida — no necesario aquí
        return_colbert_vecs=False, # Vectores ColBERT para re-ranking avanzado — no necesario aquí
    )
    # El resultado es una lista — extraemos el único vector generado
    vector_consulta = resultado['dense_vecs'][0].tolist()

    # Búsqueda por similitud en ChromaDB de top-k chunks cuyo vector sea más cercano al de la pregunta
    resultados = coleccion.query(
        query_embeddings=[vector_consulta], # ChromaDB espera lista de vectores
        n_results=top_k,
        include=[
            "documents",  # Texto del chunk
            "metadatas",  # Fuente, índice y longitud del chunk
            "distances",  # Distancia coseno (1 - similitud)
        ],
    )

    return resultados


print("Función de recuperación definida correctamente")

Función de recuperación definida correctamente


## 11. Evaluación con preguntas de complejidad progresiva

In [ ]:
# Evaluación del sistema de recuperación con las 5 preguntas
# Para cada pregunta se muestran los top-k chunks recuperados
# con su puntuación de similitud y comparación contra el ground truth
print("=" * 60)
print("EVALUACIÓN DEL SISTEMA DE RECUPERACIÓN")
print(f"Modelo: {MODELO_EMBEDDINGS} | Top-k: {TOP_K}")
print("=" * 60)

resultados_evaluacion = []

for item in PREGUNTAS_EVALUACION:
    pregunta     = item['pregunta']
    nivel        = item['nivel']
    ground_truth = item['ground_truth']

    print(f"\n[{nivel.upper()}] {pregunta}")
    print(f"Ground truth: {ground_truth}")
    print("-" * 60)

    resultados = recuperar_chunks(pregunta)

    chunks_recuperados = resultados['documents'][0]
    distancias         = resultados['distances'][0]
    # ChromaDB con métrica coseno devuelve distancias (1 - similitud)
    similitudes        = [round(1 - d, 4) for d in distancias]

    for j, (chunk, sim) in enumerate(zip(chunks_recuperados, similitudes)):
        print(f"  Chunk {j + 1} (similitud: {sim:.4f}):")
        print(f"  {chunk[:200]}...")
        print()

    resultados_evaluacion.append({
        "pregunta"    : pregunta,
        "nivel"       : nivel,
        "ground_truth": ground_truth,
        "similitudes" : similitudes,
        "chunks"      : chunks_recuperados,
    })

EVALUACIÓN DEL SISTEMA DE RECUPERACIÓN
Modelo: BAAI/bge-m3 | Top-k: 4

[BÁSICA] What metals are used in the NHC complexes studied?
Ground truth: Palladium (Pd), platinum (Pt) and gold (Au).
------------------------------------------------------------


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.99it/s]


  Chunk 1 (similitud: 0.6227):
  ## Synthesis and characterization of complexes (Pd/PtL3, PdL5/6, Pd/PtL8, Pd/AuL9)
A well-established route to obtain NHC complexes is to convert the corresponding imidazolium salts with group 10 meta...

  Chunk 2 (similitud: 0.6182):
  ## Introduction

N-heterocyclic carbenes (NHCs),  first described in 1991, 1 have found many applications. 2 There are several structural features that allow the tuning of their electronic properties....

  Chunk 3 (similitud: 0.6172):
  ## Introduction

Our group has developed several bidentate and cyclic tetradentate NHC ligands. The respective transition metal complexes have been applied e.g. in medicinal chemistry 6,7 and epoxidat...

  Chunk 4 (similitud: 0.5655):
  ## Introduction

Fig. 1 Tetracarbene ligand precursor a and derived transition metal complexes b -f .

In this study, the scope of multidentate NHC ligands is extended with an ethylene-bridged bisimid...


[BÁSICA] What is the effect of the complexes on

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.38it/s]


  Chunk 1 (similitud: 0.5969):
  ## Introduction

‡ These authors contributed equally to this work.

Synthesis, characterization, and biomedical evaluation of ethylene-bridged tetra-NHC Pd(II), Pt(II) and Au(III) complexes, with apop...

  Chunk 2 (similitud: 0.5272):
  ## Overcoming cisplatin resistance

Fig. 11 SK-N-AS and SK-N-AS cisplatin resistant cells were treated with diff erent concentrations of AuL15 and incubated for 96 h. It is shown that AuL9 was also ef...

  Chunk 3 (similitud: 0.5232):
  ## Conclusion and outlook

silver oxide is used in combination with the metal precursor and an excess of sodium acetate as a mild base, resulting in the corresponding complexes. Furthermore, the compl...

  Chunk 4 (similitud: 0.5187):
  ## Overcoming cisplatin resistance

Therefore, it is of great importance for drug development that new agents are able to overcome cytostatic drug resistance. In addition to SK-N-AS cells, AuL9 was te...


[BÁSICA] What analytical techniques were used t

...

  Chunk 1 (similitud: 0.5554):
  ## General procedures and analytical methods

Unless otherwise stated, all manipulations were performed under normal atmosphere without dried and degassed chemicals. All syntheses regarding the comple...

  Chunk 2 (similitud: 0.5408):
  ## General procedures and analytical methods

over molecular sieves (3 Å). The procedures for novel compounds obtained during the synthetic approaches to the saturated macrocyclic ligand precursor, co...

  Chunk 3 (similitud: 0.5339):
  ## General procedures and analytical methods

N 1 , N 1 , N 2 , N 2 -tetrabenzylethane-1,2-diamine, tert -butyl (2-aminoethyl)carbamate, tert -butyl 2-imidazoline-1-carboxylate) are stated in the ESI....

  Chunk 4 (similitud: 0.5150):
  ## Synthesis and characterization of complexes (Pd/PtL3, PdL5/6, Pd/PtL8, Pd/AuL9)

The absence of the acidic imidazolinium proton signal and appearance of characteristic carbene carbon signals con  r...


[INTERMEDIA] What is the role of the ethylene b

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.24it/s]


  Chunk 1 (similitud: 0.5737):
  ## Introduction

‡ These authors contributed equally to this work.

Synthesis, characterization, and biomedical evaluation of ethylene-bridged tetra-NHC Pd(II), Pt(II) and Au(III) complexes, with apop...

  Chunk 2 (similitud: 0.5727):
  ## Introduction

Fig. 1 Tetracarbene ligand precursor a and derived transition metal complexes b -f .

In this study, the scope of multidentate NHC ligands is extended with an ethylene-bridged bisimid...

  Chunk 3 (similitud: 0.5619):
  ## Introduction

Synthesis and characterization of the fi first two cyclic ethylene-bridged tetradentate NHC ligands, with an unsaturated (imidazole) and saturated backbone (2-imidazoline), are descri...

  Chunk 4 (similitud: 0.5246):
  ## Introduction

Our group has developed several bidentate and cyclic tetradentate NHC ligands. The respective transition metal complexes have been applied e.g. in medicinal chemistry 6,7 and epoxidat...


[AVANZADA] How do the cytotoxicity results of t

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  1.58it/s]

  Chunk 1 (similitud: 0.5764):
  ## Introduction

‡ These authors contributed equally to this work.

Synthesis, characterization, and biomedical evaluation of ethylene-bridged tetra-NHC Pd(II), Pt(II) and Au(III) complexes, with apop...

  Chunk 2 (similitud: 0.5217):
  ## Overcoming cisplatin resistance

Fig. 11 SK-N-AS and SK-N-AS cisplatin resistant cells were treated with diff erent concentrations of AuL15 and incubated for 96 h. It is shown that AuL9 was also ef...

  Chunk 3 (similitud: 0.5161):
  ## Conclusion and outlook

silver oxide is used in combination with the metal precursor and an excess of sodium acetate as a mild base, resulting in the corresponding complexes. Furthermore, the compl...

  Chunk 4 (similitud: 0.5138):
  ## Overcoming cisplatin resistance

Therefore, it is of great importance for drug development that new agents are able to overcome cytostatic drug resistance. In addition to SK-N-AS cells, AuL9 was te...



## 12. Resumen de resultados

In [ ]:
# Resumen cuantitativo de la evaluación
# Similitud top-1: calidad del chunk más relevante recuperado
# Similitud media: calidad global del contexto enviado al LLM
# Umbral mínimo recomendado para producción: >0.50
# El filtrado por umbral se aplicará en 04_generacion.ipynb
print("=" * 60)
print("RESUMEN — EVALUACIÓN DE RECUPERACIÓN")
print("=" * 60)
print(f"  Modelo              : {MODELO_EMBEDDINGS}")
print(f"  Top-k               : {TOP_K}")
print(f"  Documentos en índice: {coleccion.count()}")
print(f"  Preguntas evaluadas : {len(resultados_evaluacion)}")
print()
print(
    f"  {'Nivel':<12} {'Sim. top-1':>10} "
    f"{'Sim. media':>10} {'Pregunta':<35}"
)
print(f"  {'-' * 70}")

for r in resultados_evaluacion:
    sim_top1  = r['similitudes'][0]
    sim_media = round(sum(r['similitudes']) / len(r['similitudes']), 4)
    print(
        f"  {r['nivel']:<12} {sim_top1:>10.4f} "
        f"{sim_media:>10.4f} {r['pregunta'][:35]}"
    )

print("=" * 60)
print("Siguiente paso: 04_generacion.ipynb")

RESUMEN — EVALUACIÓN DE RECUPERACIÓN
  Modelo              : BAAI/bge-m3
  Top-k               : 4
  Documentos en índice: 77
  Preguntas evaluadas : 5

  Nivel        Sim. top-1 Sim. media Pregunta                           
  ----------------------------------------------------------------------
  básica           0.6227     0.6059 What metals are used in the NHC com
  básica           0.5969     0.5415 What is the effect of the complexes
  básica           0.5554     0.5363 What analytical techniques were use
  intermedia       0.5737     0.5582 What is the role of the ethylene br
  avanzada         0.5764     0.5320 How do the cytotoxicity results of 
Siguiente paso: 04_generacion.ipynb
